### Installation of Libraries & Unsloth


In [ ]:
%%capture
import os, re

if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch

    v = re.match(r"[\d]{1,}\.[\d]{1,}", str(torch.__version__)).group(0)

    xformers = "xformers==" + {
        "2.10": "0.0.34",
        "2.9": "0.0.33.post1",
        "2.8": "0.0.32.post2",
    }.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install -U ipywidgets

!pip install wandb -q


In [ ]:
import wandb

wandb.login()  # enter your own api key upon prompting


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /common/home/users/w/wlcheng.2023/.netrc.
wandb: Currently logged in as: wlunlun1212 (wlunlun1212-singapore-management-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
import os

os.environ["WANDB_PROJECT"] = "llm-financial-sentiment"  # name your project
os.environ["WANDB_LOG_MODEL"] = (
    "false"  # set to "checkpoint" if you want to save model to wandb
)


In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048  # Choose any! We auto support RoPE Scaling internally!
dtype = (
    None  # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
)
load_in_4bit = True  # Use 4bit quantization to reduce memory usage. Can be False.

# # 4bit pre quantized models we support for 4x faster downloading + no OOMs.
# fourbit_models = [
#     "unsloth/Llama-3.1-8B-bnb-4bit",      # Llama-3.1 2x faster
#     "unsloth/Llama-3.1-8B-Instruct-bnb-4bit",
#     "unsloth/Llama-3.1-70B-bnb-4bit",
#     "unsloth/Llama-3.1-405B-bnb-4bit",    # 4bit for 405b!
#     "unsloth/Mistral-Small-Instruct-2409",     # Mistral 22b 2x faster!
#     "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
#     "unsloth/Phi-3.5-mini-instruct",           # Phi-3.5 2x faster!
#     "unsloth/Phi-3-medium-4k-instruct",
#     "unsloth/gemma-2-9b-bnb-4bit",
#     "unsloth/gemma-2-27b-bnb-4bit",            # Gemma 2x faster!

#     "unsloth/Llama-3.2-1B-bnb-4bit",           # NEW! Llama 3.2 models
#     "unsloth/Llama-3.2-1B-Instruct-bnb-4bit",
#     "unsloth/Llama-3.2-3B-bnb-4bit",
#     "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",

#     "unsloth/Llama-3.3-70B-Instruct-bnb-4bit" # NEW! Llama 3.3 70B!
# ] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct",  # or choose "unsloth/Llama-3.2-1B-Instruct"
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.2.1: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    NVIDIA A40. Num GPUs = 1. Max memory: 44.424 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


# FinancialPhraseBank Dataset Loading


In [5]:
# In a Jupyter cell
!curl -L "https://raw.githubusercontent.com/maxwellsarpong/NLP-financial-text-processing-dataset/master/Sentences_75Agree.txt" -o Sentences_75Agree.txt


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  452k  100  452k    0     0   316k      0  0:00:01  0:00:01 --:--:--  316k


In [ ]:
import pandas as pd

# Load the raw text file
df = pd.read_csv(
    "Sentences_75Agree.txt",
    sep="@",  # sentence @ label
    header=None,
    names=["sentence", "label_text"],
    engine="python",
    encoding="latin-1",  # Added encoding to fix UnicodeDecodeError
)

# Strip whitespace
df["sentence"] = df["sentence"].str.strip()
df["label_text"] = df["label_text"].str.strip().str.lower()

# Map to numeric labels if you want
label_to_id = {"negative": 0, "neutral": 1, "positive": 2}
df["label"] = df["label_text"].map(label_to_id)

print(df.head(20))
print("\n======\n")
print(df["label_text"].value_counts())
print("\n======\n")
print("Total Row Count:", len(df))


                                             sentence label_text  label
0   According to Gran , the company has no plans t...    neutral      1
1   With the new production plant the company woul...   positive      2
2   For the last quarter of 2010 , Componenta 's n...   positive      2
3   In the third quarter of 2010 , net sales incre...   positive      2
4   Operating profit rose to EUR 13.1 mn from EUR ...   positive      2
5   Operating profit totalled EUR 21.1 mn , up fro...   positive      2
6   TeliaSonera TLSN said the offer is in line wit...   positive      2
7   STORA ENSO , NORSKE SKOG , M-REAL , UPM-KYMMEN...   positive      2
8   A purchase agreement for 7,200 tons of gasolin...   positive      2
9   Finnish Talentum reports its operating profit ...   positive      2
10  Clothing retail chain Sepp+ñl+ñ 's sales incre...   positive      2
11  Consolidated net sales increased 16 % to reach...   positive      2
12  Foundries division reports its sales increased...   positive

In [ ]:
# ── 80/10/10 Train / Validation / Test Split (seed=42, stratified) ─────────
# Step 1: split off 20% as temp (val + test)
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df,
    test_size=0.2,  # 20% → val + test
    random_state=42,
    stratify=df["label"],  # keeps class balance
)

# Step 2: split temp 50/50 → val (10%) and test (10%)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,  # 50% of 20% = 10% of total
    random_state=42,
    stratify=temp_df["label"],
)

print(f"Train size: {len(train_df)} (~80%)")
print(f"Val   size: {len(val_df)}  (~10%)")
print(f"Test  size: {len(test_df)}  (~10%)")

print("\n======= Class Distribution =======")
print("\nTrain distribution:")
print(train_df["label_text"].value_counts())
print("\nValidation distribution:")
print(val_df["label_text"].value_counts())
print("\nTest distribution:")
print(test_df["label_text"].value_counts())

print(
    "\n[Note] Positive examples are most frequent; neutral and negative are"
    " under-represented. This class imbalance is important context for"
    " interpreting per-class F1 scores, especially for the neutral class."
)

Train size: 2762 (~80%)
Val   size: 345  (~10%)
Test  size: 346  (~10%)

======= Class Distribution =======

Train distribution:
label_text
neutral     1717
positive     709
negative     336
Name: count, dtype: int64

Validation distribution:
label_text
neutral     214
positive     89
negative     42
Name: count, dtype: int64

Test distribution:
label_text
neutral     215
positive     89
negative     42
Name: count, dtype: int64

[Note] Positive examples are most frequent; neutral and negative are under-represented. This class imbalance is important context for interpreting per-class F1 scores, especially for the neutral class.


# Baseline Model Performance (Test Set)


In [ ]:
# Switch to Inference Mode
FastLanguageModel.for_inference(model)
model.eval()


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 3072, padding_idx=128004)
    (layers): ModuleList(
      (0): LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (k_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (up_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=3072, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((3072,)

In [ ]:
from tqdm.notebook import tqdm


def make_baseline_prompt(sentence: str) -> str:
    return tokenizer.apply_chat_template(
        [
            {
                "role": "system",
                "content": "You are a financial sentiment classifier. Reply with exactly one word: positive, negative, or neutral.",
            },
            {
                "role": "user",
                "content": f"Classify the sentiment of this financial sentence:\n{sentence}",
            },
        ],
        tokenize=False,
        add_generation_prompt=True,
    )


def run_inference(df_split, desc="Eval"):
    preds = []
    for sent in tqdm(df_split["sentence"].tolist(), desc=desc):
        prompt = make_baseline_prompt(sent)
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=16, do_sample=False)
        new_tokens = outputs[0][inputs["input_ids"].shape[1] :]
        pred = (
            tokenizer.decode(new_tokens, skip_special_tokens=True)
            .strip()
            .split("\n")[0]
        )
        preds.append(pred.lower())
    return preds


test_pred_labels = run_inference(test_df, desc="Baseline Test")

Baseline Test:   0%|          | 0/346 [00:00<?, ?it/s]

In [ ]:
def normalize(x: str) -> str:
    x = x.strip().lower()
    if "pos" in x or "bullish" in x:
        return "positive"
    if "neg" in x or "bearish" in x:
        return "negative"
    if "neu" in x or "uncertain" in x or "mixed" in x:
        return "neutral"
    for label in ["positive", "negative", "neutral"]:
        if label in x:
            return label
    return "neutral"


from sklearn.metrics import classification_report, f1_score

labels = ["negative", "neutral", "positive"]


def compute_metrics(true_texts, pred_raw, split_name="split"):
    true_norm = [t.strip().lower() for t in true_texts]
    pred_norm = [normalize(p) for p in pred_raw]
    report_text = classification_report(true_norm, pred_norm, zero_division=0)
    f1_macro = f1_score(
        true_norm, pred_norm, labels=labels, average="macro", zero_division=0
    )
    f1_neg = f1_score(
        true_norm, pred_norm, labels=["negative"], average="macro", zero_division=0
    )
    f1_neu = f1_score(
        true_norm, pred_norm, labels=["neutral"], average="macro", zero_division=0
    )
    f1_pos = f1_score(
        true_norm, pred_norm, labels=["positive"], average="macro", zero_division=0
    )
    print(f"\n=== {split_name.upper()} metrics ===")
    print(report_text)
    print(
        f"f1_macro: {f1_macro:.4f}  f1_neg: {f1_neg:.4f}  f1_neu: {f1_neu:.4f}  f1_pos: {f1_pos:.4f}"
    )
    return report_text, f1_macro, f1_neg, f1_neu, f1_pos


test_report, test_f1_macro, test_f1_neg, test_f1_neu, test_f1_pos = compute_metrics(
    test_df["label_text"].tolist(), test_pred_labels, split_name="Test"
)


=== TEST metrics ===
              precision    recall  f1-score   support

    negative       0.92      0.83      0.88        42
     neutral       0.93      0.87      0.90       215
    positive       0.75      0.90      0.82        89

    accuracy                           0.87       346
   macro avg       0.87      0.87      0.86       346
weighted avg       0.88      0.87      0.87       346

f1_macro: 0.8641  f1_neg: 0.8750  f1_neu: 0.8969  f1_pos: 0.8205


In [ ]:
import wandb

run = wandb.init(project="llm-financial-sentiment", name="base_model")
metrics = {
    "test_f1_macro": test_f1_macro,
    "test_f1_negative": test_f1_neg,
    "test_f1_neutral": test_f1_neu,
    "test_f1_positive": test_f1_pos,
}
run.log(metrics)
run.summary.update(metrics)
wandb.finish()

wandb: Detected [huggingface_hub.inference] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


test_f1_macro,▁
test_f1_negative,▁
test_f1_neutral,▁
test_f1_positive,▁
test_f1_macro,0.86413
test_f1_negative,0.875
test_f1_neutral,0.89688
test_f1_positive,0.82051


# Start Finetuning LLM + QLoRA Adaptors


In [ ]:
import os
from datasets import Dataset
from unsloth import FastLanguageModel
from unsloth.chat_templates import train_on_responses_only
from trl import SFTConfig, SFTTrainer
from transformers import DataCollatorForSeq2Seq, AutoTokenizer, EarlyStoppingCallback
from sklearn.metrics import classification_report, f1_score
from tqdm.notebook import tqdm
import torch, pandas as pd, wandb

max_seq_length = 2048
MODEL_NAME = "unsloth/Llama-3.2-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def normalize(text: str) -> str:
    t = text.strip().lower()
    if "positive" in t:
        return "positive"
    if "negative" in t:
        return "negative"
    if "neutral" in t:
        return "neutral"
    return t


def _make_chat(sentence, label=None):
    messages = [
        {
            "role": "system",
            "content": "You are a financial sentiment classifier. Reply with exactly one word: positive, negative, or neutral.",
        },
        {
            "role": "user",
            "content": f"Classify the sentiment of this financial sentence:\n{sentence}",
        },
    ]
    if label is not None:
        messages.append({"role": "assistant", "content": label})
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=(label is None),
    )


# Build HF datasets
hf_train_dataset = Dataset.from_dict(
    {
        "text": [
            _make_chat(r["sentence"], r["label_text"]) for _, r in train_df.iterrows()
        ]
    }
)
hf_val_dataset = Dataset.from_dict(
    {"text": [_make_chat(r["sentence"], r["label_text"]) for _, r in val_df.iterrows()]}
)
print(
    f"Train: {len(hf_train_dataset)} | Val: {len(hf_val_dataset)} | Test: {len(test_df)}"
)


def build_model(r: int, lora_alpha: int):
    model, tok = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=max_seq_length,
        dtype=None,
        load_in_4bit=True,
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=r,
        lora_alpha=lora_alpha,
        target_modules=[
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
            "gate_proj",
            "up_proj",
            "down_proj",
        ],
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=3407,
    )
    return model, tok


def _run_inference(model, tok, df_split, desc="Eval"):
    preds = []
    for sent in tqdm(df_split["sentence"].tolist(), desc=desc):
        prompt = _make_chat(sent)  # no label → add_generation_prompt=True
        inputs = tok(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=16, do_sample=False)
        new_tokens = out[0][inputs["input_ids"].shape[1] :]
        pred = tok.decode(new_tokens, skip_special_tokens=True).strip().split("\n")[0]
        preds.append(pred.lower())
    return preds


def _compute_split_metrics(true_texts, pred_raw):
    _labels = ["negative", "neutral", "positive"]
    true_norm = [t.strip().lower() for t in true_texts]
    pred_norm = [normalize(p) for p in pred_raw]
    return {
        "report": classification_report(true_norm, pred_norm, zero_division=0),
        "f1_macro": f1_score(
            true_norm, pred_norm, labels=_labels, average="macro", zero_division=0
        ),
        "f1_negative": f1_score(
            true_norm, pred_norm, labels=["negative"], average="macro", zero_division=0
        ),
        "f1_neutral": f1_score(
            true_norm, pred_norm, labels=["neutral"], average="macro", zero_division=0
        ),
        "f1_positive": f1_score(
            true_norm, pred_norm, labels=["positive"], average="macro", zero_division=0
        ),
    }


def evaluate_and_log(model, tok, config_name: str, train_loss: float, cfg: dict):
    FastLanguageModel.for_inference(model)
    val_preds = _run_inference(model, tok, val_df, desc=f"Val {config_name}")
    val_m = _compute_split_metrics(val_df["label_text"].tolist(), val_preds)
    print(f"\n=== {config_name} — VALIDATION ===\n{val_m['report']}")

    # ❌ Removed: test_preds, test_m, test inference

    run = wandb.init(
        project="llm-financial-sentiment",
        name=config_name,
        config=cfg,
        reinit="finish_previous",
    )
    metrics = {
        "train_loss": train_loss,
        "val_f1_macro": val_m["f1_macro"],
        "val_f1_negative": val_m["f1_negative"],
        "val_f1_neutral": val_m["f1_neutral"],
        "val_f1_positive": val_m["f1_positive"],
    }
    run.log(metrics)
    run.summary.update(metrics)
    run.finish()

    return {
        "config": config_name,
        "train_loss": round(train_loss, 4),
        "val_f1_macro": round(val_m["f1_macro"], 4),
        "val_f1_neg": round(val_m["f1_negative"], 4),
        "val_f1_neu": round(val_m["f1_neutral"], 4),
        "val_f1_pos": round(val_m["f1_positive"], 4),
    }


Train: 2762 | Val: 345 | Test: 346


In [ ]:
all_results = []

configs = [
    {
        "name": "C1_default",
        "lr": 2e-4,
        "r": 16,
        "alpha": 32,
        "batch": 4,
        "grad_acc": 1,
        "warmup": 5,
        "wd": 0.01,
    },
    {
        "name": "C2_lower_lr",
        "lr": 1e-4,
        "r": 16,
        "alpha": 32,
        "batch": 4,
        "grad_acc": 1,
        "warmup": 5,
        "wd": 0.01,
    },
    {
        "name": "C3_higher_rank",
        "lr": 2e-4,
        "r": 32,
        "alpha": 32,
        "batch": 4,
        "grad_acc": 1,
        "warmup": 5,
        "wd": 0.01,
    },
    {
        "name": "C4_higher_alpha",
        "lr": 2e-4,
        "r": 16,
        "alpha": 64,
        "batch": 4,
        "grad_acc": 1,
        "warmup": 5,
        "wd": 0.01,
    },
    {
        "name": "C5_larger_batch",
        "lr": 2e-4,
        "r": 16,
        "alpha": 32,
        "batch": 8,
        "grad_acc": 1,
        "warmup": 5,
        "wd": 0.01,
    },
    {
        "name": "C6_grad_acc",
        "lr": 2e-4,
        "r": 16,
        "alpha": 32,
        "batch": 4,
        "grad_acc": 4,
        "warmup": 5,
        "wd": 0.01,
    },
    {
        "name": "C7_longer_warmup",
        "lr": 2e-4,
        "r": 16,
        "alpha": 32,
        "batch": 4,
        "grad_acc": 1,
        "warmup": 50,
        "wd": 0.01,
    },
    {
        "name": "C8_higher_wd",
        "lr": 2e-4,
        "r": 16,
        "alpha": 32,
        "batch": 4,
        "grad_acc": 1,
        "warmup": 5,
        "wd": 0.1,
    },
]


all_results = []

for cfg in configs:
    print(f"\n{'=' * 60}\n▶ Running {cfg['name']}\n{'=' * 60}")

    try:
        model_c, tok_c = build_model(cfg["r"], cfg["alpha"])

        trainer = SFTTrainer(
            model=model_c,
            tokenizer=tok_c,
            train_dataset=hf_train_dataset,
            eval_dataset=hf_val_dataset,
            dataset_text_field="text",
            max_seq_length=max_seq_length,
            data_collator=DataCollatorForSeq2Seq(tokenizer=tok_c),
            packing=False,
            callbacks=[
                EarlyStoppingCallback(
                    early_stopping_patience=3,
                    early_stopping_threshold=0.001,
                )
            ],
            args=SFTConfig(
                per_device_train_batch_size=cfg["batch"],
                gradient_accumulation_steps=cfg["grad_acc"],
                warmup_steps=cfg["warmup"],
                num_train_epochs=1,
                learning_rate=cfg["lr"],
                logging_steps=10,
                optim="adamw_8bit",
                weight_decay=cfg["wd"],
                lr_scheduler_type="linear",
                seed=3407,
                output_dir=f"outputs/{cfg['name']}",
                report_to="none",  # Trainer touches no wandb state
                eval_strategy="steps",
                eval_steps=100,
                save_strategy="steps",
                save_steps=100,
                save_total_limit=1,
                load_best_model_at_end=True,
                metric_for_best_model="eval_loss",
                greater_is_better=False,
                dataset_num_proc=1,  # prevent multiprocessing hang
            ),
        )

        trainer = train_on_responses_only(
            trainer,
            instruction_part="<|start_header_id|>user<|end_header_id|>\n\n",
            response_part="<|start_header_id|>assistant<|end_header_id|>\n\n",
        )

        train_res = trainer.train()

        model_c.save_pretrained(f"qlora_adapters/{cfg['name']}")
        tok_c.save_pretrained(f"qlora_adapters/{cfg['name']}")
        print(f"✅ Adapter saved → qlora_adapters/{cfg['name']}")

        # ✅ wandb only opened here, after Trainer is completely done
        result = evaluate_and_log(
            model_c, tok_c, cfg["name"], train_res.training_loss, cfg
        )
        all_results.append(result)
        print(f"✅ W&B logged for {cfg['name']}")

    except Exception as e:
        print(f"❌ {cfg['name']} failed: {e}")
        import traceback

        traceback.print_exc()
        all_results.append({"config": cfg["name"], "error": str(e)})

    finally:
        if wandb.run is not None:
            wandb.finish()
        try:
            del model_c, tok_c, trainer
        except NameError:
            pass
        import gc

        gc.collect()
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        print(f"🧹 VRAM cleared after {cfg['name']}")

summary_df = pd.DataFrame(all_results)
print("\n=== Summary of All Configs ===")
print(summary_df.to_string(index=False))
summary_df.to_csv("all_results.csv", index=False)
print("Saved → all_results.csv")



▶ Running C1_default
==((====))==  Unsloth 2026.2.1: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    NVIDIA A40. Num GPUs = 1. Max memory: 44.424 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth 2026.2.1 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Unsloth: Tokenizing ["text"] (num_proc=1):   0%|          | 0/2762 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=1):   0%|          | 0/345 [00:00<?, ? examples/s]

Map (num_proc=64):   0%|          | 0/2762 [00:00<?, ? examples/s]

...

✅ Adapter saved → qlora_adapters/C8_higher_wd


Val C8_higher_wd:   0%|          | 0/345 [00:00<?, ?it/s]


=== C8_higher_wd — VALIDATION ===
              precision    recall  f1-score   support

    negative       0.95      0.90      0.93        42
     neutral       0.96      0.96      0.96       214
    positive       0.93      0.94      0.94        89

    accuracy                           0.95       345
   macro avg       0.95      0.94      0.94       345
weighted avg       0.95      0.95      0.95       345



train_loss,▁
val_f1_macro,▁
val_f1_negative,▁
val_f1_neutral,▁
val_f1_positive,▁
train_loss,0.17981
val_f1_macro,0.94192
val_f1_negative,0.92683
val_f1_neutral,0.96037
val_f1_positive,0.93855


✅ W&B logged for C8_higher_wd
🧹 VRAM cleared after C8_higher_wd

=== Summary of All Configs ===
          config  train_loss  val_f1_macro  val_f1_neg  val_f1_neu  val_f1_pos
      C1_default      0.1406        0.9385      0.9136      0.9615      0.9405
     C2_lower_lr      0.1175        0.9261      0.9268      0.9464      0.9050
  C3_higher_rank      0.1490        0.9568      0.9524      0.9700      0.9480
 C4_higher_alpha      0.1810        0.9447      0.9512      0.9616      0.9212
 C5_larger_batch      0.1126        0.9539      0.9383      0.9704      0.9529
     C6_grad_acc      0.1010        0.9465      0.9213      0.9652      0.9529
C7_longer_warmup      0.1539        0.9538      0.9524      0.9679      0.9412
    C8_higher_wd      0.1798        0.9419      0.9268      0.9604      0.9385
Saved → all_results.csv


In [ ]:
# Optuna Test
import optuna
import gc, torch, wandb, os
from unsloth import FastLanguageModel
from unsloth.chat_templates import train_on_responses_only
from trl import SFTConfig, SFTTrainer
from transformers import DataCollatorForSeq2Seq, EarlyStoppingCallback

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"


def objective(trial):
    # ── 1. Search Space (based on C1-C8 configs) ─────────────────
    lr = trial.suggest_categorical("lr", [1e-4, 2e-4])
    r = trial.suggest_categorical("r", [16, 32])
    alpha = trial.suggest_categorical("alpha", [32, 64])
    batch = trial.suggest_categorical("batch", [4, 8])
    grad_acc = trial.suggest_categorical("grad_acc", [1, 4])
    warmup = trial.suggest_categorical("warmup", [5, 50])
    wd = trial.suggest_categorical("wd", [0.01, 0.1])

    cfg_name = f"optuna_trial_{trial.number}"
    cfg = {
        "lr": lr,
        "r": r,
        "alpha": alpha,
        "batch": batch,
        "grad_acc": grad_acc,
        "warmup": warmup,
        "wd": wd,
    }

    print(f"\n{'=' * 60}\n▶ Optuna Trial {trial.number}\n{'=' * 60}")
    print(
        f"  lr={lr} | r={r} | alpha={alpha} | batch={batch} | "
        f"grad_acc={grad_acc} | warmup={warmup} | wd={wd}"
    )

    try:
        # ── 2. Build Model (same as your build_model fn) ──────────
        model_c, tok_c = build_model(r, alpha)

        # ── 3. Trainer (mirrors your original loop exactly) ───────
        trainer = SFTTrainer(
            model=model_c,
            tokenizer=tok_c,
            train_dataset=hf_train_dataset,
            eval_dataset=hf_val_dataset,
            dataset_text_field="text",
            max_seq_length=max_seq_length,
            data_collator=DataCollatorForSeq2Seq(tokenizer=tok_c),
            packing=False,
            callbacks=[
                EarlyStoppingCallback(
                    early_stopping_patience=3,
                    early_stopping_threshold=0.001,
                )
            ],
            args=SFTConfig(
                per_device_train_batch_size=batch,
                gradient_accumulation_steps=grad_acc,
                warmup_steps=warmup,
                num_train_epochs=1,
                learning_rate=lr,
                logging_steps=10,
                optim="adamw_8bit",
                weight_decay=wd,
                lr_scheduler_type="linear",
                seed=3407,
                output_dir=f"outputs/{cfg_name}",
                report_to="none",
                eval_strategy="steps",
                eval_steps=100,
                save_strategy="steps",
                save_steps=100,
                save_total_limit=1,
                load_best_model_at_end=True,
                metric_for_best_model="eval_loss",
                greater_is_better=False,
                dataset_num_proc=1,
            ),
        )

        trainer = train_on_responses_only(
            trainer,
            instruction_part="<|start_header_id|>user<|end_header_id|>\n\n",
            response_part="<|start_header_id|>assistant<|end_header_id|>\n\n",
        )

        # ── 4. Train ──────────────────────────────────────────────
        train_res = trainer.train()

        # ── 5. Save Adapter ───────────────────────────────────────
        model_c.save_pretrained(f"qlora_adapters/{cfg_name}")
        tok_c.save_pretrained(f"qlora_adapters/{cfg_name}")
        print(f"✅ Adapter saved → qlora_adapters/{cfg_name}")

        # ── 6. Evaluate on Val (same as evaluate_and_log) ─────────
        result = evaluate_and_log(
            model_c, tok_c, cfg_name, train_res.training_loss, cfg
        )
        val_f1 = result["val_f1_macro"]  # ← this is what Optuna optimizes

        print(f"✅ Trial {trial.number} Val F1 Macro: {val_f1:.4f}")
        return val_f1  # ← Optuna maximizes this

    except Exception as e:
        print(f"❌ Trial {trial.number} failed: {e}")
        import traceback

        traceback.print_exc()
        return 0.0  # failed trial gets worst score

    finally:
        if wandb.run is not None:
            wandb.finish()
        try:
            del model_c, tok_c, trainer
        except NameError:
            pass
        gc.collect()
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        print(f"🧹 VRAM cleared after trial {trial.number}")


# ── Create Study & Run ────────────────────────────────────────────────────────
study = optuna.create_study(
    direction="maximize",  # maximize val macro F1
    sampler=optuna.samplers.TPESampler(seed=42),  # Bayesian (TPE), reproducible
)

study.optimize(objective, n_trials=10)

# ── Final Results ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("=== OPTUNA COMPLETE ===")
print("=" * 60)
print(f"\n🏆 Best Trial : #{study.best_trial.number}")
print(f"   Val F1 Macro: {study.best_trial.value:.4f}")
print(f"   Best Params : {study.best_trial.params}")

optuna_df = study.trials_dataframe()
optuna_df = optuna_df.sort_values("value", ascending=False)
print("\n=== All Trials (sorted by Val F1 Macro) ===")
print(
    optuna_df[
        [
            "number",
            "value",
            "params_lr",
            "params_r",
            "params_alpha",
            "params_batch",
            "params_grad_acc",
            "params_warmup",
            "params_wd",
        ]
    ].to_string(index=False)
)

optuna_df.to_csv("optuna_results.csv", index=False)
print("Saved → optuna_results.csv")

[I 2026-02-24 21:34:54,442] A new study created in memory with name: no-name-a1a53de7-902c-4eb2-beba-c66263df03d4



▶ Optuna Trial 0
  lr=0.0002 | r=16 | alpha=32 | batch=8 | grad_acc=4 | warmup=50 | wd=0.01
==((====))==  Unsloth 2026.2.1: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    NVIDIA A40. Num GPUs = 1. Max memory: 44.424 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth: Tokenizing ["text"] (num_proc=1):   0%|          | 0/2762 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=1):   0%|          | 0/345 [00:00<?, ? examples/s]

Map (num_proc=64):   0%|          | 0/2762 [00:00<?, ? examples/s]

...

Val optuna_trial_9:   0%|          | 0/345 [00:00<?, ?it/s]


=== optuna_trial_9 — VALIDATION ===
              precision    recall  f1-score   support

    negative       0.97      0.93      0.95        42
     neutral       0.94      1.00      0.97       214
    positive       1.00      0.89      0.94        89

    accuracy                           0.96       345
   macro avg       0.97      0.94      0.95       345
weighted avg       0.96      0.96      0.96       345



train_loss,▁
val_f1_macro,▁
val_f1_negative,▁
val_f1_neutral,▁
val_f1_positive,▁
train_loss,0.12069
val_f1_macro,0.95329
val_f1_negative,0.95122
val_f1_neutral,0.96818
val_f1_positive,0.94048


[I 2026-02-24 22:18:03,060] Trial 9 finished with value: 0.9533 and parameters: {'lr': 0.0002, 'r': 32, 'alpha': 32, 'batch': 8, 'grad_acc': 1, 'warmup': 50, 'wd': 0.1}. Best is trial 0 with value: 0.9572.


✅ Trial 9 Val F1 Macro: 0.9533
🧹 VRAM cleared after trial 9

=== OPTUNA COMPLETE ===

🏆 Best Trial : #0
   Val F1 Macro: 0.9572
   Best Params : {'lr': 0.0002, 'r': 16, 'alpha': 32, 'batch': 8, 'grad_acc': 4, 'warmup': 50, 'wd': 0.01}

=== All Trials (sorted by Val F1 Macro) ===
 number  value  params_lr  params_r  params_alpha  params_batch  params_grad_acc  params_warmup  params_wd
      0 0.9572     0.0002        16            32             8                4             50       0.01
      8 0.9566     0.0001        32            32             4                4              5       0.10
      9 0.9533     0.0002        32            32             8                1             50       0.10
      6 0.9518     0.0002        16            32             8                1              5       0.01
      2 0.9443     0.0001        16            64             4                1              5       0.10
      3 0.9419     0.0002        32            64             4               

# Finetuned LLM Performance (Test Set)


In [ ]:
# ── Evaluate Test Set on a Specific Config ──────────────────────────────────
BEST_CONFIG = "C3_higher_rank"  # change to whichever config won on val F1

# Load the saved adapter
model_test, tok_test = FastLanguageModel.from_pretrained(
    model_name=f"qlora_adapters/{BEST_CONFIG}",
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model_test)
model_test.eval()

# Run inference on test set
test_preds = _run_inference(model_test, tok_test, test_df, desc=f"Test {BEST_CONFIG}")

# Compute and print metrics
test_m = _compute_split_metrics(test_df["label_text"].tolist(), test_preds)
print(f"\n=== {BEST_CONFIG} — TEST SET ===\n{test_m['report']}")
print(
    f"f1_macro: {test_m['f1_macro']:.4f}  f1_neg: {test_m['f1_negative']:.4f}  "
    f"f1_neu: {test_m['f1_neutral']:.4f}  f1_pos: {test_m['f1_positive']:.4f}"
)

# Log to wandb
run = wandb.init(project="llm-financial-sentiment", name=f"{BEST_CONFIG}_test_eval")
run.log(
    {
        "test_f1_macro": test_m["f1_macro"],
        "test_f1_negative": test_m["f1_negative"],
        "test_f1_neutral": test_m["f1_neutral"],
        "test_f1_positive": test_m["f1_positive"],
    }
)
run.summary.update({"test_f1_macro": test_m["f1_macro"]})
wandb.finish()

# Cleanup
del model_test, tok_test
gc.collect()
torch.cuda.empty_cache()

==((====))==  Unsloth 2026.2.1: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    NVIDIA A40. Num GPUs = 1. Max memory: 44.424 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Test C3_higher_rank:   0%|          | 0/346 [00:00<?, ?it/s]


=== C3_higher_rank — TEST SET ===
              precision    recall  f1-score   support

    negative       1.00      0.95      0.98        42
     neutral       0.98      0.98      0.98       215
    positive       0.95      0.97      0.96        89

    accuracy                           0.97       346
   macro avg       0.98      0.97      0.97       346
weighted avg       0.97      0.97      0.97       346

f1_macro: 0.9709  f1_neg: 0.9756  f1_neu: 0.9814  f1_pos: 0.9556


test_f1_macro,▁
test_f1_negative,▁
test_f1_neutral,▁
test_f1_positive,▁
test_f1_macro,0.97085
test_f1_negative,0.97561
test_f1_neutral,0.9814
test_f1_positive,0.95556
